In [ ]:
!pip install accelerate>=0.25.0 \
    "draccus @ git+https://github.com/dlwh/draccus" \
    einops \
    huggingface_hub \
    jsonlines \
    rich \
    sentencepiece \
    timm==0.9.10 \
    "transformers>=4.38.1" \
    opencv-python \
    ipykernel \
    seaborn \
    matplotlib \
    protobuf


In [ ]:
import os
import sys
sys.path.append("./models")
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
from PIL import Image

import torch
import torch.nn.functional as F

from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import process_images, tokenizer_image_token, get_model_name_from_path

from utils import (
    load_image,
    aggregate_llm_attention, aggregate_vit_attention,
    heterogenous_stack,
    show_mask_on_image
)

Chartgemma

In [ ]:
import os
import warnings
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig
import torch


def load_pretrained_model(model_path, model_base=None, model_name="chartgemma", load_8bit=False, load_4bit=False, device_map="auto", device="cuda", use_flash_attn=False, **kwargs):
 
    kwargs = {"device_map": device_map, **kwargs}

    if device != "cuda":
        kwargs['device_map'] = {"": device}

    # Configure quantization
    if load_8bit:
        kwargs['load_in_8bit'] = True
    elif load_4bit:
        kwargs['load_in_4bit'] = True
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
    else:
        kwargs['torch_dtype'] = torch.float16

    # Flash attention support for PaliGemma
    if use_flash_attn:
        kwargs['attn_implementation'] = 'flash_attention_2'
    kwargs['attn_implementation'] = 'eager'

    print(f'Loading ChartGemma model from {model_path}...')

    # Load processor (combines tokenizer and image processor for PaliGemma)
    processor = AutoProcessor.from_pretrained(model_path)

    # Load model
    if model_base is not None:
        # Handle LoRA or fine-tuned models
        from peft import PeftModel
        print(f'Loading base model from {model_base}...')
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            model_base,
            low_cpu_mem_usage=True,
            **kwargs
        )
        print(f"Loading adapter weights from {model_path}")
        model = PeftModel.from_pretrained(model, model_path)
        print(f"Merging weights...")
        model = model.merge_and_unload()
        print('Model loaded and merged successfully')
    else:
        # Load full model directly
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            model_path,
            low_cpu_mem_usage=True,
            **kwargs
        )
        print('Model loaded successfully')

    # Get context length
    if hasattr(model.config, "max_position_embeddings"):
        context_len = model.config.max_position_embeddings
    elif hasattr(model.config, "text_config") and hasattr(model.config.text_config, "max_position_embeddings"):
        context_len = model.config.text_config.max_position_embeddings
    else:
        context_len = 2048  # Default fallback

    return processor, model, context_len




In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
import json


model_path = "ahmed-masry/chartgemma"

# Load the model
load_8bit = False
load_4bit = False
device = "cuda" if torch.cuda.is_available() else "cpu"

processor, model, context_len = load_pretrained_model(
    model_path,
    None,  # model_base
    "chartgemma",
    load_8bit,
    load_4bit,
    device=device
)

# Get the number of image tokens (patches)
vision_model = model.vision_tower if hasattr(model, 'vision_tower') else model.vision_model
if hasattr(vision_model.config, 'num_image_tokens'):
    num_image_tokens = vision_model.config.num_image_tokens
else:
    image_size = vision_model.config.image_size
    patch_size = vision_model.config.patch_size
    num_image_tokens = (image_size // patch_size) ** 2

# Calculate grid size for attention visualization
grid_size = int(np.sqrt(num_image_tokens))

# Create output directory for visualizations
os.makedirs("attention_outputs", exist_ok=True)
qapairs_path = "/content/image_indices.json"
images_dir = "/content/pngval/png"
# Process each question
with open(qapairs_path, 'r') as f:
    qa_pairs = json.load(f)
qa_pairs = qa_pairs[:50]  # Only first 50

for idx, qa in enumerate(qa_pairs):
    image_index = qa['image_index']
    prompt_text = qa['question_string']
    chart_type = qa['type']
    answer = qa['answer']
    image_path = os.path.join(images_dir, f"{image_index}.png")
    
    print(f"\n{'='*80}")
    print(f"Processing {idx + 1}/{len(qa_pairs)}: {chart_type}")
    print(f"Question: {prompt_text}")
    print(f"Image: {image_path}")
    print('='*80)
    
    # Load image
    try:
        image = load_image(image_path)
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        continue
    
    # Prepare inputs
    inputs = processor(text=prompt_text, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate response with attention
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=512,
            use_cache=True,
            return_dict_in_generate=True,
            output_attentions=True,
        )
    
    # Decode the response
    response = processor.decode(outputs["sequences"][0], skip_special_tokens=True).strip()
    print(f"\nResponse: {response}\n")
    
    # Construct attention matrix
    aggregated_prompt_attention = []
    for i, layer in enumerate(outputs["attentions"][0]):
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        cur = attns_per_head[:-1].cpu().clone()
        cur[1:, 0] = 0.
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)
        aggregated_prompt_attention.append(cur)
    aggregated_prompt_attention = torch.stack(aggregated_prompt_attention).mean(dim=0)
    
    # Build LLM attention matrix
    llm_attn_matrix = heterogenous_stack(
        [torch.tensor([1])]
        + list(aggregated_prompt_attention)
        + list(map(aggregate_llm_attention, outputs["attentions"]))
    )
    
    # Calculate token positions
    total_sequence_len = outputs["sequences"].shape[1]
    input_prompt_len = inputs["input_ids"].shape[1]
    
    vision_token_start = 0
    vision_token_end = num_image_tokens
    
    output_token_start = input_prompt_len
    output_token_len = total_sequence_len - input_prompt_len
    output_token_end = total_sequence_len
    
    # Validate
    if output_token_len <= 0:
        print(f"Warning: No new tokens generated for question {idx + 1}")
        continue
    
    # Create visualization
    num_image_per_row = 8
    image_ratio = image.size[0] / image.size[1]
    num_rows = output_token_len // num_image_per_row + (1 if output_token_len % num_image_per_row != 0 else 0)
    
    if num_rows == 0:
        num_rows = 1
    
    fig, axes = plt.subplots(
        num_rows, num_image_per_row,
        figsize=(10, (10 / num_image_per_row) * image_ratio * num_rows),
        dpi=150
    )
    
    if num_rows == 1:
        axes = axes.reshape(1, -1)
    
    plt.subplots_adjust(wspace=0.05, hspace=0.2)
    
    vis_overlayed_with_attn = True
    output_token_inds = list(range(output_token_start, output_token_end))
    vision_model = model.vision_tower if hasattr(model, 'vision_tower') else model.vision_model

    # Get vision attention from all layers
    if hasattr(vision_model, 'image_attentions'):
        vision_attentions = vision_model.image_attentions
    else:
        # Forward pass on vision model to get attentions
        with torch.no_grad():
            vision_outputs = vision_model(
                pixel_values=inputs["pixel_values"],
                output_attentions=True
            )
            vision_attentions = vision_outputs.attentions

    # Aggregate vision attention from ALL layers (changed from last layer only)
    print(f"Aggregating attention from {len(vision_attentions)} vision encoder layers")
    
    # Stack and average attention across all layers
    all_layer_attentions = []
    for attn in vision_attentions:
        # Average across attention heads
        layer_attn = attn.mean(dim=1).squeeze(0)
        all_layer_attentions.append(layer_attn)
    
    # Stack all layers and compute mean
    stacked_attentions = torch.stack(all_layer_attentions)
    vis_attn_matrix_all_layers = stacked_attentions.mean(dim=0)
    
    # Remove CLS token attention if present (first token)
    if vis_attn_matrix_all_layers.shape[0] > num_image_tokens:
        vis_attn_matrix_all_layers = vis_attn_matrix_all_layers[1:, 1:]  # Remove CLS token
    
    # Now vis_attn_matrix_all_layers contains aggregated attention from all layers
    vis_attn_matrix = vis_attn_matrix_all_layers
    
    patch_size = vision_model.config.patch_size
    image_size_model = vision_model.config.image_size
    grid_size = image_size_model // patch_size

    for i, ax in enumerate(axes.flatten()):
      if i >= output_token_len:
        ax.axis("off")
        continue
    
      target_token_ind = output_token_inds[i]
      attn_weights_over_vis_tokens = llm_attn_matrix[target_token_ind][vision_token_start:vision_token_end]
    
      if len(attn_weights_over_vis_tokens) == 0 or attn_weights_over_vis_tokens.sum() == 0:
        ax.axis("off")
        continue
    
    # Normalize attention weights
      attn_weights_over_vis_tokens = attn_weights_over_vis_tokens / attn_weights_over_vis_tokens.sum()
    
    # DIRECTLY reshape to spatial grid - no vision self-attention needed
      attn_over_image = attn_weights_over_vis_tokens.reshape(grid_size, grid_size)
      attn_over_image = attn_over_image / attn_over_image.max()
    
    # Interpolate to image size
      attn_over_image = F.interpolate(
        attn_over_image.unsqueeze(0).unsqueeze(0),
        size=(image.size[1], image.size[0]),
        mode='bilinear',
        align_corners=False
      ).squeeze()
    
    # Visualize
      np_img = np.array(image)[:, :, ::-1]
      img_with_attn, heatmap = show_mask_on_image(np_img, attn_over_image.cpu().numpy())
      ax.imshow(heatmap if not vis_overlayed_with_attn else img_with_attn)
    
      token_id = outputs["sequences"][0][target_token_ind]
      ax.set_title(
        processor.decode([token_id], skip_special_tokens=False).strip(),
        fontsize=7,
        pad=1
      )
      ax.axis("off")
    # Save visualization
    output_filename = f"attention_outputs/{chart_type}_{image_index}.svg"
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close()
    
    print(f"Saved visualization to: {output_filename}")

print(f"\n{'='*80}")
print(f"Completed processing 50 questions")
print(f"Visualizations saved in: attention_outputs/")
print('='*80)

Llava

In [ ]:
import os
import warnings
import shutil

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
import torch
from llava.model import *
from llava.constants import DEFAULT_IMAGE_PATCH_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN


def load_pretrained_model1(model_path, model_base, model_name, load_8bit=False, load_4bit=False, device_map="auto", device="cuda", use_flash_attn=False, **kwargs):
    kwargs = {"device_map": device_map, **kwargs}

    if device != "cuda":
        kwargs['device_map'] = {"": device}

    if load_8bit:
        kwargs['load_in_8bit'] = True
    elif load_4bit:
        kwargs['load_in_4bit'] = True
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4'
        )
    else:
        kwargs['torch_dtype'] = torch.float16

    if use_flash_attn:
        kwargs['attn_implementation'] = 'flash_attention_2'
    else:
    # Force eager attention to enable attention output
      kwargs['attn_implementation'] = 'eager'
    print(f"kwargs before loading model: {kwargs}")
    if 'llava' in model_name.lower():
        # Load LLaVA model
        if 'lora' in model_name.lower() and model_base is None:
            warnings.warn('There is `lora` in model name but no `model_base` is provided. If you are loading a LoRA model, please provide the `model_base` argument. Detailed instruction: https://github.com/haotian-liu/LLaVA#launch-a-model-worker-lora-weights-unmerged.')
        if 'lora' in model_name.lower() and model_base is not None:
            from llava.model.language_model.llava_llama import LlavaConfig
            lora_cfg_pretrained = LlavaConfig.from_pretrained(model_path)
            tokenizer = AutoTokenizer.from_pretrained(model_base, use_fast=False)
            print('Loading LLaVA from base model...')
            model = LlavaLlamaForCausalLM.from_pretrained(model_base, low_cpu_mem_usage=True, config=lora_cfg_pretrained, **kwargs)
            token_num, tokem_dim = model.lm_head.out_features, model.lm_head.in_features
            if model.lm_head.weight.shape[0] != token_num:
                model.lm_head.weight = torch.nn.Parameter(torch.empty(token_num, tokem_dim, device=model.device, dtype=model.dtype))
                model.model.embed_tokens.weight = torch.nn.Parameter(torch.empty(token_num, tokem_dim, device=model.device, dtype=model.dtype))

            print('Loading additional LLaVA weights...')
            if os.path.exists(os.path.join(model_path, 'non_lora_trainables.bin')):
                non_lora_trainables = torch.load(os.path.join(model_path, 'non_lora_trainables.bin'), map_location='cpu')
            else:
                # this is probably from HF Hub
                from huggingface_hub import hf_hub_download
                def load_from_hf(repo_id, filename, subfolder=None):
                    cache_file = hf_hub_download(
                        repo_id=repo_id,
                        filename=filename,
                        subfolder=subfolder)
                    return torch.load(cache_file, map_location='cpu')
                non_lora_trainables = load_from_hf(model_path, 'non_lora_trainables.bin')
            non_lora_trainables = {(k[11:] if k.startswith('base_model.') else k): v for k, v in non_lora_trainables.items()}
            if any(k.startswith('model.model.') for k in non_lora_trainables):
                non_lora_trainables = {(k[6:] if k.startswith('model.') else k): v for k, v in non_lora_trainables.items()}
            model.load_state_dict(non_lora_trainables, strict=False)

            from peft import PeftModel
            print('Loading LoRA weights...')
            model = PeftModel.from_pretrained(model, model_path)
            print('Merging LoRA weights...')
            model = model.merge_and_unload()
            print('Model is loaded...')
        elif model_base is not None:
            # this may be mm projector only
            print('Loading LLaVA from base model...')
            if 'mpt' in model_name.lower():
                if not os.path.isfile(os.path.join(model_path, 'configuration_mpt.py')):
                    shutil.copyfile(os.path.join(model_base, 'configuration_mpt.py'), os.path.join(model_path, 'configuration_mpt.py'))
                tokenizer = AutoTokenizer.from_pretrained(model_base, use_fast=True)
                cfg_pretrained = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
                model = LlavaMptForCausalLM.from_pretrained(model_base, low_cpu_mem_usage=True, config=cfg_pretrained, **kwargs)
            else:
                tokenizer = AutoTokenizer.from_pretrained(model_base, use_fast=False)
                cfg_pretrained = AutoConfig.from_pretrained(model_path)
                model = LlavaLlamaForCausalLM.from_pretrained(model_base, low_cpu_mem_usage=True, config=cfg_pretrained, **kwargs)

            mm_projector_weights = torch.load(os.path.join(model_path, 'mm_projector.bin'), map_location='cpu')
            mm_projector_weights = {k: v.to(torch.float16) for k, v in mm_projector_weights.items()}
            model.load_state_dict(mm_projector_weights, strict=False)
        else:
            if 'mpt' in model_name.lower():
                tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
                model = LlavaMptForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, **kwargs)
            elif 'mistral' in model_name.lower():
                tokenizer = AutoTokenizer.from_pretrained(model_path)
                model = LlavaMistralForCausalLM.from_pretrained(
                    model_path,
                    low_cpu_mem_usage=True,
                    **kwargs
                )
            else:
                tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
                model = LlavaLlamaForCausalLM.from_pretrained(
                    model_path,
                    low_cpu_mem_usage=True,
                    **kwargs
                )
    else:
        # Load language model
        if model_base is not None:
            # PEFT model
            from peft import PeftModel
            tokenizer = AutoTokenizer.from_pretrained(model_base, use_fast=False)
            model = AutoModelForCausalLM.from_pretrained(model_base, low_cpu_mem_usage=True, **kwargs)
            print(f"Loading LoRA weights from {model_path}")
            model = PeftModel.from_pretrained(model, model_path)
            print(f"Merging weights")
            model = model.merge_and_unload()
            print('Convert to FP16...')
            model.to(torch.float16)
        else:
            use_fast = False
            if 'mpt' in model_name.lower():
                tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
                model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, trust_remote_code=True, **kwargs)
            else:
                tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
                model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, **kwargs)

    image_processor = None

    if 'llava' in model_name.lower():
        mm_use_im_start_end = getattr(model.config, "mm_use_im_start_end", False)
        mm_use_im_patch_token = getattr(model.config, "mm_use_im_patch_token", True)
        if mm_use_im_patch_token:
            tokenizer.add_tokens([DEFAULT_IMAGE_PATCH_TOKEN], special_tokens=True)
        if mm_use_im_start_end:
            tokenizer.add_tokens([DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN], special_tokens=True)
        model.resize_token_embeddings(len(tokenizer))

        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            vision_tower.load_model(device_map=device_map)
        if device_map != 'auto':
            vision_tower.to(device=device_map, dtype=torch.float16)
            if hasattr(vision_tower, 'vision_tower') and hasattr(vision_tower.vision_tower, 'config'):
                vision_tower.vision_tower.config._attn_implementation = 'eager'
        image_processor = vision_tower.image_processor

    if hasattr(model.config, "max_sequence_length"):
        context_len = model.config.max_sequence_length
    else:
        context_len = 2048

    return tokenizer, model, image_processor, context_len


In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
import json


model_path = "liuhaotian/llava-v1.5-7b"

# Load the model
load_8bit = False
load_4bit = False
device = "cuda" if torch.cuda.is_available() else "cpu"

disable_torch_init()

model_name = get_model_name_from_path(model_path)
tokenizer, model, image_processor, context_len = load_pretrained_model1(
    model_path,
    None,  # model_base
    model_name,
    load_8bit,
    load_4bit,
    device_map='cuda',
    device=device,
    use_flash_attn=False
)

print(f"Model config _attn_implementation: {getattr(model.config, '_attn_implementation', 'not set')}")
print(f"Model config _attn_implementation_internal: {getattr(model.config, '_attn_implementation_internal', 'not set')}")

# Check the language model specifically (LLaVA wraps a language model)
if hasattr(model, 'model'):
    print(f"Language model config: {getattr(model.model.config, '_attn_implementation', 'not set')}")

# Get grid size
grid_size = model.get_vision_tower().num_patches_per_side

# Create output directory for visualizations
os.makedirs("attention_outputs", exist_ok=True)

# Determine conversation mode
if "llama-2" in model_name.lower():
    conv_mode = "llava_llama_2"
elif "mistral" in model_name.lower():
    conv_mode = "mistral_instruct"
elif "v1.6-34b" in model_name.lower():
    conv_mode = "chatml_direct"
elif "v1" in model_name.lower():
    conv_mode = "llava_v1"
elif "mpt" in model_name.lower():
    conv_mode = "mpt"
else:
    conv_mode = "llava_v0"
qapairs_path = "/content/image_indices.json"
images_dir = "/content/pngval/png"
# Process each question
with open(qapairs_path, 'r') as f:
    qa_pairs = json.load(f)
qa_pairs = qa_pairs[:50]  # Only first 50

for idx, qa in enumerate(qa_pairs):
    image_index = qa['image_index']
    prompt_text = qa['question_string']
    chart_type = qa['type']
    answer = qa['answer']
    image_path = os.path.join(images_dir, f"{image_index}.png")
    
    print(f"\n{'='*80}")
    print(f"Processing {idx + 1}/{len(qa_pairs)}: {chart_type}")
    print(f"Question: {prompt_text}")
    print(f"Image: {image_path}")
    print('='*80)
    
    # Load image
    try:
        image = load_image(image_path)
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        continue
    
    # Process image
    image_tensor, images = process_images([image], image_processor, model.config)
    image = images[0]
    image_size = image.size
    if type(image_tensor) is list:
        image_tensor = [img.to(model.device, dtype=torch.float16) for img in image_tensor]
    else:
        image_tensor = image_tensor.to(model.device, dtype=torch.float16)
    
    # Prepare conversation
    conv = conv_templates[conv_mode].copy()
    if "mpt" in model_name.lower():
        roles = ('user', 'assistant')
    else:
        roles = conv.roles
    
    # Construct input
    if model.config.mm_use_im_start_end:
        inp = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + prompt_text
    else:
        inp = DEFAULT_IMAGE_TOKEN + '\n' + prompt_text
    
    conv.append_message(conv.roles[0], inp)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()
    
    # Remove system prompt to avoid attention bias
    prompt = prompt.replace(
        "A chat between a curious human and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions. ",
        ""
    )
    
    # Tokenize
    input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(model.device)
    
    # Generate with attention
    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            images=image_tensor,
            image_sizes=[image_size],
            do_sample=False,
            max_new_tokens=512,
            use_cache=True,
            return_dict_in_generate=True,
            output_attentions=True,
        )
    
    # Decode response
    response = tokenizer.decode(outputs["sequences"][0], skip_special_tokens=True).strip()
    print(f"\nResponse: {response}\n")
    
    # Aggregate prompt attention
    aggregated_prompt_attention = []
    for i, layer in enumerate(outputs["attentions"][0]):
        layer_attns = layer.squeeze(0)
        attns_per_head = layer_attns.mean(dim=0)
        cur = attns_per_head[:-1].cpu().clone()
        cur[1:, 0] = 0.
        cur[1:] = cur[1:] / cur[1:].sum(-1, keepdim=True)
        aggregated_prompt_attention.append(cur)
    aggregated_prompt_attention = torch.stack(aggregated_prompt_attention).mean(dim=0)
    
    print(f"Number of layers in prompt attention: {len(outputs['attentions'][0])}")
    print(f"Aggregated prompt attention shape: {aggregated_prompt_attention.shape}")
    
    # Build LLM attention matrix
    llm_attn_matrix = heterogenous_stack(
        [torch.tensor([1])]
        + list(aggregated_prompt_attention)
        + list(map(aggregate_llm_attention, outputs["attentions"]))
    )
    
    print(f"LLM attention matrix shape: {llm_attn_matrix.shape}")
    
    # Calculate token positions
    num_vision_patches = model.get_vision_tower().num_patches
    
    # Vision tokens replace the <image> token in the sequence
    vision_token_start = len(tokenizer(prompt.split("<image>")[0], return_tensors='pt')["input_ids"][0])
    vision_token_end = vision_token_start + num_vision_patches
    
    # Input length in attention matrix
    input_token_len = num_vision_patches + len(input_ids[0]) - 1
    
    # The number of NEW tokens generated (not including prompt)
    # outputs["attentions"] has one entry per generated token (excluding prompt)
    num_generated_tokens = len(outputs["attentions"])
    
    # Total length in attention matrix
    total_attn_len = llm_attn_matrix.shape[0]
    
    # Generated tokens in the output sequence
    # outputs["sequences"] contains: prompt tokens + generated tokens
    # We need to extract just the generated portion
    full_sequence = outputs["sequences"][0]
    
    # The generated tokens are the last num_generated_tokens in the sequence
    generated_tokens = full_sequence[-num_generated_tokens:] if num_generated_tokens > 0 else torch.tensor([])
    
    # Positions in attention matrix for generated tokens
    output_token_start = input_token_len
    output_token_end = total_attn_len
    output_token_len = num_generated_tokens
    
    # Debug: Print token information
    print(f"Vision tokens: {vision_token_start} to {vision_token_end} (count: {num_vision_patches})")
    print(f"Input token length in attn matrix: {input_token_len}")
    print(f"Total attention matrix length: {total_attn_len}")
    print(f"Number of generated tokens: {num_generated_tokens}")
    print(f"Output sequence length: {len(full_sequence)}")
    print(f"Generated token range in attn matrix: {output_token_start} to {output_token_end}")
    if num_generated_tokens > 0:
        print(f"Generated tokens IDs: {generated_tokens.tolist()}")
        print(f"Generated text: {tokenizer.decode(generated_tokens, skip_special_tokens=True)}")
    
    # Validate
    if output_token_len <= 0:
        print(f"Warning: No new tokens generated for question {idx + 1}")
        continue
    
    # Create visualization
    num_image_per_row = 8
    image_ratio = image_size[0] / image_size[1]
    num_rows = output_token_len // num_image_per_row + (1 if output_token_len % num_image_per_row != 0 else 0)
    
    if num_rows == 0:
        num_rows = 1
    
    fig, axes = plt.subplots(
        num_rows, num_image_per_row,
        figsize=(16, (16 / num_image_per_row) * image_ratio * num_rows),
        dpi=150
    )
    
    if num_rows == 1:
        axes = axes.reshape(1, -1)
    
    plt.subplots_adjust(wspace=0.1, hspace=0.3, top=0.95, bottom=0.05)
    
    vis_overlayed_with_attn = True
    output_token_inds = list(range(output_token_start, output_token_end))
    
    # CORRECTED ATTENTION VISUALIZATION - NO VISION SELF-ATTENTION
    for i, ax in enumerate(axes.flatten()):
        if i >= output_token_len:
            ax.axis("off")
            continue
        
        target_token_ind = output_token_inds[i]
        
        # Check if index is valid
        if target_token_ind >= llm_attn_matrix.shape[0]:
            print(f"Warning: token index {target_token_ind} out of bounds, skipping")
            ax.axis("off")
            continue
            
        attn_weights_over_vis_tokens = llm_attn_matrix[target_token_ind][vision_token_start:vision_token_end]
        
        if len(attn_weights_over_vis_tokens) == 0 or attn_weights_over_vis_tokens.sum() == 0:
            print(f"Warning: No attention to vision tokens for token {i}")
            ax.axis("off")
            continue
        
        # Normalize attention weights
        attn_weights_over_vis_tokens = attn_weights_over_vis_tokens / attn_weights_over_vis_tokens.sum()
        
        # Debug: Check attention distribution
        if i == 0:
            print(f"Attention stats - min: {attn_weights_over_vis_tokens.min():.6f}, max: {attn_weights_over_vis_tokens.max():.6f}, mean: {attn_weights_over_vis_tokens.mean():.6f}")
        
        # DIRECT RESHAPE - NO VISION SELF-ATTENTION LOOP
        attn_over_image = attn_weights_over_vis_tokens.reshape(grid_size, grid_size)
        
        # Normalize BEFORE max division to see full range
        attn_over_image = attn_over_image / attn_over_image.max()
        
        # Interpolate to image size using bilinear interpolation
        attn_over_image = F.interpolate(
            attn_over_image.unsqueeze(0).unsqueeze(0),
            size=image_size,
            mode='bilinear',
            align_corners=False
        ).squeeze()
        
        # Visualize
        np_img = np.array(image)[:, :, ::-1]
        img_with_attn, heatmap = show_mask_on_image(np_img, attn_over_image.cpu().numpy())
        ax.imshow(heatmap if not vis_overlayed_with_attn else img_with_attn)
        
        # Set title with decoded token - FIXED TITLE VISIBILITY
        if i < len(generated_tokens):
            token_id = generated_tokens[i]
            token_text = tokenizer.decode([token_id], skip_special_tokens=False).strip()
            ax.set_title(
                token_text if token_text else f"[ID:{token_id}]",
                fontsize=9,
                pad=3,
                color='black',
                backgroundcolor='white',
                weight='bold'
            )
        ax.axis("off")
    
    # Save visualization
    output_filename = f"lava_out_attn/{chart_type}_{image_index}.svg"
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close()
    
    print(f"Saved visualization to: {output_filename}")

print(f"\n{'='*80}")
print(f"Completed processing 50 questions")
print(f"Visualizations saved in: lava_out_attn/")
print('='*80)